In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import cross_validate
from sklearn.multiclass import OneVsOneClassifier
from sklearn.multiclass import OneVsRestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.utils import resample

df = pd.read_csv('../winequality-red-clean.csv', delimiter=',')
df.info()

In [ ]:

'''
Selected Features classification:
 Index(['volatile acidity', 'citric acid', 'total sulfur dioxide', 'sulphates',
       'alcohol'], dtype='object')
'''

selected_features = ['volatile acidity', 'citric acid', 'total sulfur dioxide', 'sulphates',
                     'alcohol']
X_selected = df[selected_features]
y = df['quality']

standard_scaler = StandardScaler()
X_selected = standard_scaler.fit_transform(X_selected)


In [ ]:
def calculate_bias_variance(model, X, y, num_samples=100):
    # Array to store predictions from all bootstrapped models
    all_predictions = np.zeros((num_samples, X.shape[0]))

    # Perform bootstrapping
    for i in range(num_samples):
        # Resample the training data (bootstrap sample)
        X_resampled, y_resampled = resample(X, y)

        # Train the model on the resampled data
        model.fit(X_resampled, y_resampled)

        # Make predictions on the entire dataset (both train and test data)
        all_predictions[i, :] = model.predict(X)

    # Calculate the mean prediction for each data point
    mean_predictions = np.mean(all_predictions, axis=0)

    # Calculate the bias^2: The squared difference between true values and mean predictions
    bias_squared = np.mean((mean_predictions - y) ** 2)

    # Calculate the variance: The average variance of predictions across different bootstrapped models
    variance = np.mean(np.var(all_predictions, axis=0))
    
    print(f"Bias^2: {bias_squared}")
    print(f"Variance: {variance}")

In [ ]:
def train_test_accuracy(train_accuracy, test_accuracy):
    print(f'Training Accuracy: {(train_accuracy * 100).__round__(3)}%')
    print(f'Testing Accuracy: {(test_accuracy * 100).__round__(3)}%')

In [ ]:
param_grid = {
    'estimator__C': [0.001, 0.01, 0.1, 1, 10, 100, 1000]
}

# Initialize the OneVsRestClassifier with a linear SVM as the base model
ovr_linear_model = OneVsRestClassifier(SVC(kernel='linear'))

# Initialize GridSearchCV to perform hyperparameter tuning
grid_search = GridSearchCV(estimator=ovr_linear_model, param_grid=param_grid, cv=5, scoring='accuracy', verbose=1)

# Perform the grid search with cross-validation
grid_search.fit(X_selected, y)

# Extract the best model and its corresponding parameters
best_model = grid_search.best_estimator_
best_params = grid_search.best_params_
best_score = grid_search.best_score_

# Output the best parameters and the corresponding accuracy
print(f"Best Parameters: {best_params}")
print(f"Best Cross-Validation Accuracy: {best_score:.4f}")

# You can also perform cross-validation with the best model to check the bias-variance tradeoff
cv_results = cross_validate(best_model, X_selected, y, cv=5, return_train_score=True)
train_accuracy = cv_results['train_score'].mean()
test_accuracy = cv_results['test_score'].mean()

print(f'OneVsRestClassifier with Best Parameters:')
train_test_accuracy(train_accuracy, test_accuracy)
calculate_bias_variance(best_model,X_selected,y)

'''
Fitting 5 folds for each of 7 candidates, totalling 35 fits
Best Parameters: {'estimator__C': 1000}
Best Cross-Validation Accuracy: 0.5378

'''


In [ ]:
# Define the parameter grid for hyperparameter tuning
param_grid_rbf = {
    'estimator__C': [0.001, 0.01, 0.1, 1, 10, 100, 1000],  # Regularization parameter
    'estimator__gamma': [0.001, 0.01, 0.1, 1, 10]  # Gamma parameter for the RBF kernel
}

# Initialize the OneVsRestClassifier with an SVM using the RBF kernel as the base model
ovr_rbf_model = OneVsRestClassifier(SVC(kernel='rbf'))

# Initialize GridSearchCV to perform hyperparameter tuning
grid_search = GridSearchCV(estimator=ovr_rbf_model, param_grid=param_grid_rbf, cv=5, scoring='accuracy', verbose=1)

# Perform the grid search with cross-validation
grid_search.fit(X_selected, y)

# Extract the best model and its corresponding parameters
best_model = grid_search.best_estimator_
best_params = grid_search.best_params_
best_score = grid_search.best_score_

# Output the best parameters and the corresponding accuracy
print(f"Best Parameters: {best_params}")
print(f"Best Cross-Validation Accuracy: {best_score:.4f}")

# You can also perform cross-validation with the best model to check the bias-variance tradeoff
cv_results = cross_validate(best_model, X_selected, y, cv=5, return_train_score=True)
train_accuracy = cv_results['train_score'].mean()
test_accuracy = cv_results['test_score'].mean()

print(f'OneVsRestClassifier RBF with Best Parameters:')
train_test_accuracy(train_accuracy, test_accuracy)
calculate_bias_variance(best_model,X_selected,y)

'''
Best Parameters: {'estimator__C': 1, 'estimator__gamma': 0.1}
'''

In [ ]:
# Define the parameter grid for hyperparameter tuning (varying C values)
param_grid = {
    'estimator__C': [0.001, 0.01, 0.1, 1, 10, 100, 1000]  # Values for the regularization parameter C
}

# Initialize the OneVsOneClassifier with a linear SVM as the base model
ovo_linear_model = OneVsOneClassifier(SVC(kernel='linear'))

# Initialize GridSearchCV to perform hyperparameter tuning
grid_search = GridSearchCV(estimator=ovo_linear_model, param_grid=param_grid, cv=5, scoring='accuracy', verbose=1)

# Perform the grid search with cross-validation
grid_search.fit(X_selected, y)

# Extract the best model and its corresponding parameters
best_model = grid_search.best_estimator_
best_params = grid_search.best_params_
best_score = grid_search.best_score_

# Output the best parameters and the corresponding accuracy
print(f"Best Parameters: {best_params}")
print(f"Best Cross-Validation Accuracy: {best_score:.4f}")

# You can also perform cross-validation with the best model to check the bias-variance tradeoff
cv_results = cross_validate(best_model, X_selected, y, cv=5, return_train_score=True)
train_accuracy = cv_results['train_score'].mean()
test_accuracy = cv_results['test_score'].mean()

print(f'OneVsOneClassifier with Best Parameters:')
train_test_accuracy(train_accuracy, test_accuracy)
calculate_bias_variance(best_model,X_selected,y)

'''
Best Parameters: {'estimator__C': 10}
'''

In [ ]:
# Define the parameter grid for hyperparameter tuning
param_grid_rbf = {
    'estimator__C': [0.001, 0.01, 0.1, 1, 10, 100, 1000],  # Regularization parameter
    'estimator__gamma': [0.001, 0.01, 0.1, 1, 10]  # Gamma parameter for the RBF kernel
}

# Initialize the OneVsOneClassifier with an SVM using the RBF kernel as the base model
ovo_rbf_model = OneVsOneClassifier(SVC(kernel='rbf'))

# Initialize GridSearchCV to perform hyperparameter tuning
grid_search = GridSearchCV(estimator=ovo_rbf_model, param_grid=param_grid_rbf, cv=5, scoring='accuracy', verbose=1)

# Perform the grid search with cross-validation
grid_search.fit(X_selected, y)

# Extract the best model and its corresponding parameters
best_model = grid_search.best_estimator_
best_params = grid_search.best_params_
best_score = grid_search.best_score_

# Output the best parameters and the corresponding accuracy
print(f"Best Parameters: {best_params}")
print(f"Best Cross-Validation Accuracy: {best_score:.4f}")

# You can also perform cross-validation with the best model to check the bias-variance tradeoff
cv_results = cross_validate(best_model, X_selected, y, cv=5, return_train_score=True)
train_accuracy = cv_results['train_score'].mean()
test_accuracy = cv_results['test_score'].mean()

print(f'OneVsOneClassifier RBF with Best Parameters:')
train_test_accuracy(train_accuracy, test_accuracy)
calculate_bias_variance(best_model,X_selected,y)

'''
Best Parameters: {'estimator__C': 1, 'estimator__gamma': 0.1}
'''